In [6]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder # what is meesage place holder ??
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory #in built memory in langchain --
from langchain_core.runnables.history import RunnableWithMessageHistory # wait for 15 mint
from langchain_core.messages import HumanMessage, AIMessage,BaseMessage # by default your LLM try --> input --> human or AI meeage
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from pydantic import BaseModel, Field
from typing import List


/Users/rahultiwari/Documents/02_Freelancing/coding_ninja_fresh/dummy-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
load_dotenv()

True

In [8]:

class WindowChatMessageHistory(BaseChatMessageHistory, BaseModel):
    """
    A chat message history that only keeps the last `k` messages.
    
    When the history exceeds k messages, the OLDEST messages
    are silently dropped. This creates a sliding window effect.
    
    Implements BaseChatMessageHistory interface so it works
    seamlessly with RunnableWithMessageHistory.
    """
    messages: List[BaseMessage] = Field(default_factory=list)
    k: int = Field(default=6)  # keep last k messages (= k/2 turns)

    def add_messages(self, messages: List[BaseMessage]) -> None:
        """
        Add new messages, then trim to last k.
        Called automatically by RunnableWithMessageHistory.
        """
        self.messages.extend(messages)
        
        # Trim: keep only the last k messages
        # k=6 means 3 HumanMessages + 3 AIMessages = 3 turns
        if len(self.messages) > self.k:
            dropped = len(self.messages) - self.k
            self.messages = self.messages[-self.k:]
            print(f"  [Window] Dropped {dropped} oldest message(s). Now: {len(self.messages)} messages.")

    def clear(self) -> None:
        """Clear all messages."""
        self.messages = []







In [10]:
llm = ChatOpenAI(model="gpt-4o-mini")

In [11]:

WINDOW_K = 4  # keep last 4 messages = 2 full turns (human + AI each)

prompt = ChatPromptTemplate.from_messages([
    ("system", """\
You are a professional email support agent.
Classify: Billing / Technical / General. Priority: High / Medium / Low.
Remember customer details within the conversation."""),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

chain = prompt | llm | StrOutputParser()

In [12]:
window_store = {}

def get_session_history(session_id):
    if session_id not in window_store:
        window_store[session_id] = WindowChatMessageHistory(k=WINDOW_K)
    return window_store[session_id]

In [13]:
message = ['abc','xyz','abc','I am ok']
updated_list = []
k = 2
if len(message)>k:
    message = message[k:]

print(message)

['abc', 'I am ok']


In [14]:
get_session_history("test")

WindowChatMessageHistory(messages=[], k=4)

In [15]:
window_assistant = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

In [16]:
cfg = {"configurable": {"session_id": "window_john"}}


In [17]:
window_assistant.invoke({"input":"Billing complaint from John — invoice #1042, overcharged $250."},
                        config=cfg)

'Classification: Billing  \nPriority: High  \n\nCustomer details: John, invoice #1042, overcharged $250.'

In [18]:
turns = [
    "Billing complaint from John — invoice #1042, overcharged $250.",
    "What category is his complaint?",
    "What is the SLA for billing?",         # ← Turn 3 — Turn 1 gets dropped after this
    "Can you remind me who made the original complaint?",  # ← CRITICAL: Turn 1 is now GONE
    "Draft a 2-line apology email for him.",
]


for i, turn in enumerate(turns, 1):
    response = window_assistant.invoke({"input": turn}, config=cfg)
    hist = window_store["window_john"]
    print(response)

Classification: Billing  
Priority: High  

Customer details: John, invoice #1042, overcharged $250.
  [Window] Dropped 2 oldest message(s). Now: 4 messages.
John's complaint falls under the category of Billing.
  [Window] Dropped 2 oldest message(s). Now: 4 messages.
The typical Service Level Agreement (SLA) for billing inquiries is usually 48 hours for a response. However, specific SLAs may vary depending on the company's policies. It's best to check the specific SLA provided by the organization you are referring to for accurate details.
  [Window] Dropped 2 oldest message(s). Now: 4 messages.
The original complaint was made by John.
  [Window] Dropped 2 oldest message(s). Now: 4 messages.
Subject: Our Sincere Apologies

Dear John,  
We sincerely apologize for any inconvenience you've experienced and appreciate your patience as we address your concerns. Thank you for bringing this to our attention.  

Best regards,  
[Your Name]  
[Your Position]  


In [7]:
message = ['2','3']

k=2
if len(message)>k:
    print(message[2:])
